In [ ]:
import numpy as np
from heapq import heappush, heappop
from math import sqrt
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import Point
from queue import PriorityQueue
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import LineString


def read_raster(file_path):
    with rasterio.open(file_path) as src:
        return src.read(1), src.transform
def heuristic(current, end):
    """使用欧氏距离作为启发式函数"""
    return sqrt((end[0] - current[0]) ** 2 + (end[1] - current[1]) ** 2)

def get_neighbors(position, shape):
    """获取八方向邻居节点"""
    directions = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    x, y = position
    neighbors = []
    for dx, dy in directions:
        nx, ny = x + dx, y + dy
        if 0 <= nx < shape[0] and 0 <= ny < shape[1]:
            neighbors.append((nx, ny))
    return neighbors

def astar_algorithm(mesc, erv, start, end):
    open_set = []
    heappush(open_set, (0 + heuristic(start, end), start, [start], mesc[start]))

    visited = set()
    visited.add(start)

    while open_set:
        _, current, path, mesc_val = heappop(open_set)

        if current == end:
            return path, mesc_val, 'Success'

        for neighbor in get_neighbors(current, mesc.shape):
            if neighbor in visited:
                continue
            
            new_path = path + [neighbor]
            length = len(new_path)
            new_mesc_val = (mesc_val * (length - 1) + mesc[neighbor]) / length

            cost = new_mesc_val - mesc_val + heuristic(neighbor, end)
            heappush(open_set, (cost, neighbor, new_path, new_mesc_val))
            visited.add(neighbor)

    return [], 0, 'Failure'  

points_of_interests = pd.read_csv(r"F:\xx\ecological opt.csv")
points_of_interests = pd.DataFrame(np.concatenate([points_of_interests[["start", "startX", "startY"]].values, points_of_interests[["end", "endX", "endY"]].values], axis=0))
points_of_interests.columns = ["idx", "X", "Y"]
points_of_interests = points_of_interests.drop_duplicates().astype(int).reset_index(drop=True)
mapping_dict = {}
for idx, row in points_of_interests.iterrows():
    mapping_dict[int(row["idx"])] = (row["X"], row["Y"])

points_of_interests = pd.read_csv("F:\xx\ecological opt.csv")
points_of_interests

In [ ]:
final_input = points_of_interests[["rank", "start", "end"]].astype(int)
final_input["start_cord"] = final_input["start"].map(mapping_dict) 
final_input["end_cord"] = final_input["end"].map(mapping_dict) 
final_input

In [ ]:
mesc_data, _ = read_raster(r'F:\XX能\Ecological importance.tif')
erv_data, _ = read_raster(r'F:\xx\MCR.tif')

allpaths=[]
for start, end in zip(final_input["start_cord"], final_input["end_cord"]):
    path, avg_mesc_val, status = astar_algorithm(mesc_data, erv_data, start, end)
    allpaths.append(path)
print('ok')

In [ ]:
with rasterio.open(r'F:\xx\MCR.tif') as dataset:
    band1 = dataset.read(1)
    transform = dataset.transform
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(band1, cmap='gray', origin='upper')
sumpath=[]
for i in range(0,len(allpaths)):
    path=allpaths[i]

    if path:
        ax.plot([p[1] for p in path], [p[0] for p in path])
        sumpath.append(path)
    else:
        print(f"No path between {start} and {end}")

gdf_paths = gpd.GeoDataFrame(columns=['geometry'])

for path in sumpath:

    if len(path)>1:  
        geo_path=[transform * (y, x) for x, y in path]
        line = LineString(geo_path)
        gdf_paths = gdf_paths.append({'geometry': line}, ignore_index=True)

gdf_paths.crs = "EPSG:4326"  

gdf_paths.to_file(r"F:\xx\path_output_Astar.shp")
plt.show()